# Step10c — Salidas Estadísticas (Ecuador, 2015–2025)

**Proyecto:** EFCI Ecuador (ENSO Flood Chronicity Index)
**Fase:** 10 — Productos, sección C: rankings, tendencias, análisis ENSO (Fase 0 §9)
**Autor / Author:** Jerson Achicanoy, Information Management and GIS Expert
**Fecha:** 2026-08-06

**Objetivo de este notebook:** consolidar en un solo workbook Excel
(`outputs/tables/EFCI_Ecuador_Estadisticas.xlsx`) los "Statistical
Outputs" pedidos en el diseño original (Fase 0 §9): ranking de
provincias, ranking de cantones, tendencias mensuales, tendencias
anuales, y análisis por categoría ENSO — más los gráficos
correspondientes en `outputs/figures/`.

**No recalcula ningún índice**: reutiliza las salidas ya validadas de
las Fases 3, 8 y 9. Es un ejercicio de consolidación y presentación,
no de análisis nuevo.

**English:** Statistical outputs for Ecuador, 2015–2025: canton and province rankings, monthly and annual trends, ENSO category analysis, exported to a formatted Excel workbook.

## 0. Configuración del entorno

**English:** Environment setup.

In [ ]:
# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper that creates folders when missing
from src.utilidades.io_utils import ensure_dir

# Tabular data
import pandas as pd
# Numerical arrays
import numpy as np
# Vector geodata
import geopandas as gpd
# Plotting library
import matplotlib.pyplot as plt
# Excel workbook reader/writer
from openpyxl import load_workbook
# Cell styles
from openpyxl.styles import Font, PatternFill, Alignment
# Column letter helper
from openpyxl.utils import get_column_letter

# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration
config = load_config(CONFIG_PATH)
# Create the notebook logger
logger = get_logger("step10c_statistical_outputs", config, log_dir=PROJECT_ROOT / "logs")
# Log the start of the notebook
logger.info("Fase 10c iniciada — configuración cargada desde %s", CONFIG_PATH)

# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Output folder for tables
tables_dir = ensure_dir(PROJECT_ROOT / "outputs" / "tables")
# Output folder for figures
figures_dir = ensure_dir(PROJECT_ROOT / "outputs" / "figures")

# Default font size
plt.rcParams["font.size"] = 10
# Month labels for charts and tables
MES_NOMBRES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

## 1. Ranking de cantones y provincias (EFCI, Fase 8)

Se ordenan de mayor a menor EFCI, con los 4 sub-índices como columnas
de contexto (para que el ranking no sea una caja negra) — igual que en
la Fase 8 §7, pero exportado aquí en un formato tabular final, listo
para revisar sin abrir un notebook.

**English:** Canton and province ranking by chronicity index.

In [ ]:
# Read the composite index per canton
efci_canton = pd.read_csv(processed_dir / "efci_composite_canton.csv", dtype={"canton_code": str})
# Keep the combined (all hazards) view
efci_canton_todos = efci_canton[efci_canton["hazard_type"] == "TODOS"].copy()
# Canton ranking table sorted by chronicity index
ranking_cantones = efci_canton_todos[[
    "province_name_official", "canton_name_official", "n_eventos", "efci", "clase_efci",
    "severity_index", "recurrence_index", "persistence_index", "enso_magnitude_mean",
]].sort_values("efci", ascending=False).reset_index(drop=True)
# Add the rank position
ranking_cantones.insert(0, "posicion", range(1, len(ranking_cantones) + 1))
# Report column names
ranking_cantones.columns = [
    "Posición", "Provincia", "Cantón", "Nº Eventos", "EFCI", "Clase EFCI",
    "Severidad", "Recurrencia", "Persistencia", "Magnitud ENSO (media)",
]
logger.info("Ranking de cantones: %d filas", len(ranking_cantones))

# Read the composite index per province
efci_provincia = pd.read_csv(processed_dir / "efci_composite_provincia.csv", dtype={"province_code": str})
# Keep the combined view
efci_provincia_todos = efci_provincia[efci_provincia["hazard_type"] == "TODOS"].copy()
# Province ranking table
ranking_provincias = efci_provincia_todos[[
    "province_name_official", "n_eventos", "efci", "clase_efci",
]].sort_values("efci", ascending=False).reset_index(drop=True)
# Add the rank position
ranking_provincias.insert(0, "posicion", range(1, len(ranking_provincias) + 1))
# Report column names
ranking_provincias.columns = ["Posición", "Provincia", "Nº Eventos", "EFCI", "Clase EFCI"]
logger.info("Ranking de provincias: %d filas", len(ranking_provincias))

ranking_cantones.head(10)

## 2. Tendencia mensual (estacionalidad, 2015-2025 agregado)

**English:** Monthly trend (seasonality, 2015–2025 aggregated).

In [ ]:
# Read the georeferenced events
he = gpd.read_file(processed_dir / "hazard_event_geo.gpkg", layer="hazard_event")

# Events per calendar month
tendencia_mensual = he.groupby("month").size().reindex(range(1, 13), fill_value=0).reset_index()
tendencia_mensual.columns = ["Mes", "Nº Eventos (2015-2025)"]
# Add month names
tendencia_mensual.insert(1, "Nombre Mes", MES_NOMBRES)

# Desglose adicional: eventos en meses "El Niño" (cualquier magnitud, criterio de Fase 5 §4.3) vs. resto
# Flag events that happened in El Niño months
he["es_nino"] = he["icen_category"].str.startswith("El Niño", na=False)
# Events per month split by El Niño / other months
tendencia_mensual_enso = he.groupby(["month", "es_nino"]).size().unstack(fill_value=0).reindex(range(1, 13), fill_value=0)
# Column labels
tendencia_mensual_enso.columns = ["Meses no-El Niño", "Meses El Niño"]
# Join both tables
tendencia_mensual = tendencia_mensual.merge(tendencia_mensual_enso, left_on="Mes", right_index=True)

# New figure
fig, ax = plt.subplots(figsize=(9, 5))
# Bars: non-El Niño months
ax.bar(tendencia_mensual["Nombre Mes"], tendencia_mensual["Meses no-El Niño"], label="Meses no-El Niño", color="#9ecae1")
# Stacked bars: El Niño months
ax.bar(tendencia_mensual["Nombre Mes"], tendencia_mensual["Meses El Niño"],
       bottom=tendencia_mensual["Meses no-El Niño"], label="Meses El Niño", color="#de2d26")
ax.set_title("Estacionalidad de eventos (2015-2025) — total y en meses El Niño")
ax.set_ylabel("Nº de eventos")
ax.legend()
fig.tight_layout()
# Save the monthly chart
fig.savefig(figures_dir / "10c_tendencia_mensual.png", dpi=150)
logger.info("Gráfico de tendencia mensual generado")
plt.close(fig)
tendencia_mensual

## 3. Tendencia anual (2015-2025)

**English:** Annual trend (2015–2025).

In [ ]:
# Events per year
tendencia_anual = he.groupby("year").size().reset_index()
tendencia_anual.columns = ["Año", "Nº Eventos"]

# Categoría ICEN dominante del año (la más frecuente entre los 12 meses de ese año,
# tomada de la serie mensual completa de Fase 5, no solo de los meses con evento)
# Read the monthly ENSO series
enso_monthly = pd.read_csv(processed_dir / "enso_index_monthly_extended.csv")
# Dominant ICEN category of each year
categoria_dominante = enso_monthly.groupby("year")["icen_category"].agg(lambda s: s.value_counts().idxmax())
# Attach the dominant category
tendencia_anual = tendencia_anual.merge(categoria_dominante.rename("Categoría ICEN dominante"), left_on="Año", right_index=True, how="left")

fig, ax = plt.subplots(figsize=(10, 5))
# Bar colour by dominant category: red El Niño, blue La Niña, grey neutral
colores_barra = tendencia_anual["Categoría ICEN dominante"].map(
    lambda c: "#de2d26" if str(c).startswith("El Niño") else ("#3182bd" if str(c).startswith("La Niña") else "#969696")
)
ax.bar(tendencia_anual["Año"].astype(str), tendencia_anual["Nº Eventos"], color=colores_barra)
ax.set_title("Eventos por año (2015-2025) — color según categoría ICEN dominante del año")
ax.set_ylabel("Nº de eventos")
# Legend patches
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#de2d26", label="Año dominado por El Niño"),
    Patch(color="#3182bd", label="Año dominado por La Niña"),
    Patch(color="#969696", label="Año dominado por Neutro"),
])
fig.tight_layout()
# Save the annual chart
fig.savefig(figures_dir / "10c_tendencia_anual.png", dpi=150)
logger.info("Gráfico de tendencia anual generado")
plt.close(fig)
tendencia_anual

## 4. Análisis por categoría ENSO (ICEN)

**English:** Analysis by ENSO (ICEN) category.

In [ ]:
# Event statistics per ICEN category
analisis_enso = he.groupby("icen_category").agg(
    n_eventos=("event_id", "count"),
    poblacion_afectada_max=("affected_direct", "max"),
    cantones_distintos=("canton_code", "nunique"),
).reset_index()
analisis_enso.columns = ["Categoría ICEN", "Nº Eventos", "Máx. Afectados Directos (1 evento)", "Nº Cantones Distintos Afectados"]
# Sort by number of events
analisis_enso = analisis_enso.sort_values("Nº Eventos", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 5))
# Order for the horizontal bars
orden = analisis_enso.sort_values("Nº Eventos")
ax.barh(orden["Categoría ICEN"], orden["Nº Eventos"], color="#6baed6")
ax.set_title("Nº de eventos por categoría ICEN (2015-2025)")
ax.set_xlabel("Nº de eventos")
fig.tight_layout()
# Save the ENSO chart
fig.savefig(figures_dir / "10c_analisis_enso.png", dpi=150)
logger.info("Gráfico de análisis ENSO generado")
plt.close(fig)
analisis_enso

## 5. Exportación a workbook Excel

Un solo archivo con 5 hojas: Resumen, Ranking_Cantones,
Ranking_Provincias, Tendencia_Mensual, Tendencia_Anual, Analisis_ENSO
— fuente Arial, encabezados en negrita, panel superior fijo (freeze
panes) y autofiltro en las tablas largas, para que sea legible
directamente en Excel sin reformatear.

**English:** Export to an Excel workbook.

In [ ]:
# Output Excel workbook
xlsx_path = tables_dir / "EFCI_Ecuador_Estadisticas.xlsx"

# Write one sheet per table
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    # Summary sheet with project metadata
    resumen = pd.DataFrame({
        "Campo": [
            "Proyecto", "Fase", "Fecha de generación", "Autor",
            "Cantones cubiertos", "Provincias cubiertas", "Período",
            "Fórmula EFCI",
        ],
        "Valor": [
            "EFCI Ecuador (ENSO Flood Chronicity Index)",
            "Fase 10 — Productos (salidas estadísticas)",
            "2026-08-06", "Jerson Achicanoy (FAO OER)",
            221, 24, "2015-2025",
            "0.35·Recurrencia + 0.30·Severidad + 0.20·Persistencia + 0.15·Magnitud ENSO",
        ],
    })
    resumen.to_excel(writer, sheet_name="Resumen", index=False)
    ranking_cantones.to_excel(writer, sheet_name="Ranking_Cantones", index=False)
    ranking_provincias.to_excel(writer, sheet_name="Ranking_Provincias", index=False)
    tendencia_mensual.to_excel(writer, sheet_name="Tendencia_Mensual", index=False)
    tendencia_anual.to_excel(writer, sheet_name="Tendencia_Anual", index=False)
    analisis_enso.to_excel(writer, sheet_name="Analisis_ENSO", index=False)

logger.info("Workbook exportado (contenido base): %s", xlsx_path)

In [ ]:
# --- Formato profesional (fuente Arial, encabezados, freeze panes, autofiltro) ---
# Reopen the workbook to apply formatting
wb = load_workbook(xlsx_path)
# Header style: dark blue fill
HEADER_FILL = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
# Header font: bold white
HEADER_FONT = Font(name="Arial", bold=True, color="FFFFFF", size=10)
# Body font
BODY_FONT = Font(name="Arial", size=10)

# Format every sheet
for nombre_hoja in wb.sheetnames:
    ws = wb[nombre_hoja]
    for cell in ws[1]:
        cell.font = HEADER_FONT
        cell.fill = HEADER_FILL
        cell.alignment = Alignment(horizontal="center", vertical="center")
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.font = BODY_FONT
    # Ancho de columna automático (aproximado por longitud de contenido)
    for col_cells in ws.columns:
        longitud = max((len(str(c.value)) if c.value is not None else 0) for c in col_cells)
        col_letter = get_column_letter(col_cells[0].column)
        ws.column_dimensions[col_letter].width = min(max(longitud + 2, 10), 45)
    if nombre_hoja != "Resumen":
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

# Save the formatted workbook
wb.save(xlsx_path)
logger.info("Formato profesional aplicado: %s", xlsx_path)
print(f"Workbook final: {xlsx_path}")

## 6. Validación

- El workbook tiene 6 hojas y no está vacío.
- El ranking de cantones tiene 221 filas (1 por cantón, sin duplicados
  ni faltantes) y el de provincias 24.
- La suma de `Tendencia_Anual["Nº Eventos"]` coincide exactamente con
  el total de eventos del proyecto (21,743) — confirma que no se
  perdió ni duplicó ningún evento en la agregación.

**English:** Validation.

In [ ]:
# Check the workbook was written
assert xlsx_path.exists() and xlsx_path.stat().st_size > 0
# Check all 221 cantons are ranked
assert len(ranking_cantones) == 221
# canton_name_official NO es único a nivel nacional (p. ej. "Bolívar" existe como
# cantón tanto en Carchi como en Manabí, "Olmedo" en Loja y en Manabí) — la llave
# única real es (Provincia, Cantón), verificado aquí en vez de asumir unicidad del nombre solo.
# Check no duplicated (province, canton) pairs
assert not ranking_cantones.duplicated(["Provincia", "Cantón"]).any(), "hay pares (Provincia, Cantón) duplicados"
# Check all 24 provinces are ranked
assert len(ranking_provincias) == 24 and ranking_provincias["Provincia"].is_unique
# Check the annual trend adds up to the total number of events
assert tendencia_anual["Nº Eventos"].sum() == len(he), "la tendencia anual no cuadra con el total de eventos"
logger.info("Fase 10c — validación OK.")
print("Fase 10c completa.")